# Семинар 7. Семантическая сегментация Unet

### В этом задании Вам предстоит:
1. Реализовать архитектуру Unet на Keras
2. Обучить свою модель на небольшом датасете и сохранить её  

Источник: https://www.tensorflow.org/tutorials/images/segmentation

### Импорт библиотек и загрузка данных

In [ ]:
%%bash
pip install tensorflow-datasets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

In [ ]:
data, info = tfds.load('oxford_iiit_pet:4.*.*', with_info=True)
train_dataset = data['train']
test_dataset = data['test']
print(info)

### Визуализация изображений и масок

In [ ]:
def display(display_list):
  plt.figure(figsize=(15, 15))

  title = ['Input Image', 'True Mask', 'Predicted Mask']

  for i in range(len(display_list)):
    plt.subplot(1, len(display_list), i+1)
    plt.title(title[i])
    plt.imshow(tf.keras.utils.array_to_img(display_list[i]))
    plt.axis('off')
  plt.show()

In [ ]:
for entity in train_dataset.take(2):
  sample_image, sample_mask = entity['image'], entity['segmentation_mask']
  display([sample_image, sample_mask])

### Операция Conv2DTranspose (Upconvolution, upsample)
https://arxiv.org/pdf/1603.07285v1.pdf

In [ ]:
x = np.ones((1,5,5,1))
out = tf.keras.layers.Conv2DTranspose(
    filters=1,
    kernel_size=(2,2),
    strides=2,
    kernel_initializer='ones')(x)

In [ ]:
out.numpy().reshape(10,10)

### Архитектура Unet 
<img src='https://raw.githubusercontent.com/shreyaspadhy/UNet-Zoo/master/unet.png'>

С помощью Functional API можно строить более гибкие архитектуры.  
Например, несколько слоев можно объединить друг с другом с помощью обычных python - функций. 

In [ ]:
def double_conv_block(x, n_filters):
    # Conv2D then ReLU activation
    x = tf.keras.layers.Conv2D(n_filters, 
                      kernel_size=3, 
                      padding = "same", 
                      activation = "relu", 
                      kernel_initializer = "he_normal")(x)
    # Conv2D then ReLU activation
    x = tf.keras.layers.Conv2D(n_filters, 
                      kernel_size=3,
                      padding = "same",
                      activation = "relu",
                      kernel_initializer = "he_normal")(x)
    return x

def downsample_block(x, n_filters):
    conv_features = double_conv_block(x, n_filters)
    p = tf.keras.layers.MaxPool2D(2)(conv_features)
    return conv_features, p

def upsample_block(x, conv_features, n_filters):
    # upsample
    x = tf.keras.layers.Conv2DTranspose(n_filters, 
                               kernel_size=3, 
                               strides=2, padding="same")(x)
    # concatenate
    x = tf.keras.layers.concatenate([x, conv_features])
    # Conv2D twice with ReLU activation
    x = double_conv_block(x, n_filters)
    return x

### Задание 1. Реализуйте архитектуру Unet на Keras 

Не забудьте про нормализацию входных данных!

In [ ]:
 # inputs
inputs = tf.keras.layers.Input(shape=(256,256,3))

# encoder: contracting path - downsample
# downsample
f1, p1 = downsample_block(inputs, 256)

# bottleneck
bottleneck = double_conv_block(p1, 1024)

# decoder: expanding path - upsample
#  upsample
u1 = upsample_block(bottleneck, f1, 256)

# outputs
outputs = tf.keras.layers.Conv2D(3, 1, padding="same", activation = "softmax")(u1)

# unet model with Keras Functional API
unet_model = tf.keras.Model(inputs, outputs, name="U-Net")

In [ ]:
unet_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

### Не забываем проверить модель на малой выборке данных 

In [ ]:
@tf.function
def resize_image_and_mask(example):
    image = tf.image.resize(example['image'], (256, 256))
    segmentation_mask =  tf.image.resize(example['segmentation_mask'], (256, 256)) - 1
    return image, segmentation_mask


In [ ]:
# resize and batch dataset  
small_data = train_dataset.map(resize_image_and_mask).take(2).batch(2).cache()
model_history = unet_model.fit(small_data, epochs=4)

### Вопрос 1. 

Чему равно матожидание начального значения segmentation CE_loss?  
**Ответ:** 

### Задание 2. Обучите Unet и сохраните модель 

In [ ]:
unet_model = tf.keras.Model(inputs, outputs, name="U-Net")
unet_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [ ]:
# todo read the docs
# save every epoch 
model_history = unet_model.fit(
    train_dataset.map(resize_image_and_mask).cache().batch(4).prefetch(tf.data.AUTOTUNE), 
    epochs=10,
    callbacks=tf.keras.callbacks.ModelCheckpoint("models/unet_pet_model.keras", save_best_only=True),
    validation_data=test_dataset.map(resize_image_and_mask).cache().batch(4).prefetch(tf.data.AUTOTUNE))

In [ ]:
loss = model_history.history['loss']
val_loss = model_history.history['val_loss']

plt.figure()
plt.plot(model_history.epoch, loss, 'r', label='Training loss')
plt.plot(model_history.epoch, val_loss, 'bo', label='Validation loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss Value')
plt.ylim([0, 1])
plt.legend()
plt.show()

### Задание 3. Визуализируйте предсказанные маски